In [4]:
import re
import os
import glob
from collections import OrderedDict
from datetime import datetime

def extract_test_results(dataset, model, task_name, log_dir):
    """
    从log文件中提取测试结果
    
    Args:
        dataset: 数据集名称
        model: 模型名称  
        task_name: 任务名称
        log_dir: log文件目录路径
    
    Returns:
        dict: 提取的测试结果，如果未找到则返回None
    """
    
    # 构建文件模式
    file_pattern = os.path.join(log_dir, dataset, model, f"{model}-{dataset}-{task_name}-*.log")
    
    # 获取匹配的文件列表
    log_files = glob.glob(file_pattern)
    filtered_files = []
    for file in log_files:
        filename = os.path.basename(file)
        # 精确匹配模式：model-dataset-task_name-Jul或Aug
        if (f"-{task_name}-Jul-" in filename or f"-{task_name}-Aug-" in filename):
            filtered_files.append(file)

    log_files = filtered_files
    
    if not log_files:
        # print(f"未找到匹配的log文件: {file_pattern}")
        return None
    
    # 按文件修改时间排序，最新的在前
    log_files.sort(key=lambda x: os.path.getmtime(x), reverse=True)
    
    # 正则表达式匹配测试结果
    test_result_pattern = r"test result: OrderedDict\(\[(.*?)\]\)"
    metric_pattern = r"\('(\w+)', np\.float64\(([\d.]+)\)\)"
    
    # 从最新的文件开始查找
    for log_file in log_files:
        try:
            with open(log_file, 'r', encoding='utf-8') as f:
                content = f.read()
                
            # 查找测试结果行
            match = re.search(test_result_pattern, content)
            if match:
                metrics_str = match.group(1)
                
                # 提取各个指标
                metrics = {}
                for metric_match in re.finditer(metric_pattern, metrics_str):
                    metric_name = metric_match.group(1)
                    metric_value = float(metric_match.group(2))
                    if metric_name == 'auc':
                        metric_value *= 100  # 将AUC值转换为百分比
                        metric_value = round(metric_value, 2)  # 保留两位小数
                    metrics[metric_name] = metric_value
                
                if metrics:
                    # print(f"从文件 {log_file} 中找到测试结果:")
                    # print(f"Test Results: {metrics}")
                    return metrics
                    
        except Exception as e:
            print(f"读取文件 {log_file} 时出错: {e}")
            continue
    
    # print(f"未在任何匹配的log文件中找到测试结果")
    return None

def extract_test_results_simple(log_file_path):
    """
    从单个log文件中提取测试结果的简化版本
    
    Args:
        log_file_path: log文件路径
    
    Returns:
        dict: 提取的测试结果
    """
    
    test_result_pattern = r"test result: OrderedDict\(\[(.*?)\]\)"
    metric_pattern = r"\('(\w+)', np\.float64\(([\d.]+)\)\)"
    
    try:
        with open(log_file_path, 'r', encoding='utf-8') as f:
            content = f.read()
            
        match = re.search(test_result_pattern, content)
        if match:
            metrics_str = match.group(1)
            metrics = {}
            
            for metric_match in re.finditer(metric_pattern, metrics_str):
                metric_name = metric_match.group(1)
                metric_value = float(metric_match.group(2))
                if metric_name == 'auc':
                    metric_value *= 100  # 将AUC值转换为百分比
                    metric_value = round(metric_value, 2)  # 保留两位小数
                metrics[metric_name] = metric_value
            
            return metrics
            
    except Exception as e:
        print(f"处理文件时出错: {e}")
        return None

# 使用示例
if __name__ == "__main__":
    # 方法1: 根据数据集、模型、任务名自动查找
    dataset = "m4a"
    model = "AFM" 
    task_name = "all-t-mulan16"
    log_dir = "/home/zhangyk/zhouyz/rec/recbole_v2/log"
    
    results = extract_test_results(dataset, model, task_name, log_dir)
    
    # 方法2: 直接处理指定文件
    log_file = "/home/zhangyk/zhouyz/rec/recbole_v2/log/m4a/AFM/AFM-m4a-all-t-mulan16-Jul-30-2025_22-22-18-fb8124.log"
    results = extract_test_results_simple(log_file)
    
    if results:
        print("提取的测试结果:")
        for metric, value in results.items():
            print(f"{metric}: {value}")

提取的测试结果:
auc: 82.85
logloss: 0.3445
ndcg: 1.0


In [49]:
dataset = "m4a"
model = "FinalMLP" 
task_name = "id-a-t"
log_dir = "/home/zhangyk/zhouyz/rec/recbole_v2/log"
    
results = extract_test_results(dataset, model, task_name, log_dir)
results
    

{'auc': 81.69, 'logloss': 0.3542, 'ndcg': 1.0}

In [6]:
dataset = "lfm2b-fil"
task_name = "idonly"
models = ["LR", "FM", "FFM", "AFM", "FiGNN", "WideDeep", "NFM", "DeepFM", "xDeepFM", "AutoInt", "DCN", "DCNV2", "MaskNet","FinalMLP", "EulerNet", "WuKong"]
for model in models:
    results = extract_test_results(dataset, model, task_name, log_dir)
    if results:
        print(f"Model: {model}, Results: {results}")
    else:
        print(f"Model: {model}, No results found.")

Model: LR, Results: {'auc': 81.27, 'logloss': 0.436, 'ndcg': 0.863}
Model: FM, Results: {'auc': 85.17, 'logloss': 0.4194, 'ndcg': 0.7799}
Model: FFM, Results: {'auc': 84.06, 'logloss': 0.4321, 'ndcg': 1.0}
Model: AFM, Results: {'auc': 85.12, 'logloss': 0.4223, 'ndcg': 1.0}
Model: FiGNN, No results found.
Model: WideDeep, Results: {'auc': 84.02, 'logloss': 0.4142, 'ndcg': 1.0}
Model: NFM, Results: {'auc': 83.32, 'logloss': 0.4601, 'ndcg': 0.6628}
Model: DeepFM, Results: {'auc': 83.81, 'logloss': 0.6279, 'ndcg': 1.0}
Model: xDeepFM, Results: {'auc': 82.03, 'logloss': 0.4289, 'ndcg': 0.89}
Model: AutoInt, Results: {'auc': 84.63, 'logloss': 0.4113, 'ndcg': 0.8522}
Model: DCN, Results: {'auc': 85.22, 'logloss': 0.397, 'ndcg': 1.0}
Model: DCNV2, Results: {'auc': 85.6, 'logloss': 0.3946, 'ndcg': 1.0}
Model: MaskNet, Results: {'auc': 84.78, 'logloss': 0.4032, 'ndcg': 0.6285}
Model: FinalMLP, Results: {'auc': 85.79, 'logloss': 0.4021, 'ndcg': 1.0}
Model: EulerNet, Results: {'auc': 85.01, 'loglo

In [61]:
dataset = "m4a"
task_names = ["token-t-cluster8", "token-t-cluster8", "token-t-cluster16", "token-t-cluster", "token-t-cluster64"]
models = ["LR", "FM", "FFM", "AFM", "FiGNN", "WideDeep", "NFM", "DeepFM", "xDeepFM", "AutoInt", "DCN", "DCNV2", "MaskNet","FinalMLP", "EulerNet", "WuKong"]
for task_name in task_names:
    print(f"Task: {task_name}")
    for model in models:
        results = extract_test_results(dataset, model, task_name, log_dir)
        if results:
            print(f"Model: {model}, Results: {results}")
        else:
            print(f"Model: {model}, No results found.")

Task: token-t-cluster8
Model: LR, No results found.
Model: FM, Results: {'auc': 80.26, 'logloss': 0.3662, 'ndcg': 1.0}
Model: FFM, No results found.
Model: AFM, Results: {'auc': 82.98, 'logloss': 0.3472, 'ndcg': 1.0}
Model: FiGNN, No results found.
Model: WideDeep, Results: {'auc': 81.91, 'logloss': 0.3788, 'ndcg': 1.0}
Model: NFM, Results: {'auc': 82.77, 'logloss': 0.3481, 'ndcg': 1.0}
Model: DeepFM, Results: {'auc': 81.43, 'logloss': 0.3553, 'ndcg': 1.0}
Model: xDeepFM, Results: {'auc': 82.3, 'logloss': 0.3498, 'ndcg': 1.0}
Model: AutoInt, Results: {'auc': 77.91, 'logloss': 0.3728, 'ndcg': 0.0694}
Model: DCN, Results: {'auc': 82.66, 'logloss': 0.3499, 'ndcg': 1.0}
Model: DCNV2, Results: {'auc': 83.14, 'logloss': 0.3482, 'ndcg': 1.0}
Model: MaskNet, Results: {'auc': 83.57, 'logloss': 0.3393, 'ndcg': 1.0}
Model: FinalMLP, Results: {'auc': 81.78, 'logloss': 0.3629, 'ndcg': 1.0}
Model: EulerNet, Results: {'auc': 82.55, 'logloss': 0.3601, 'ndcg': 1.0}
Model: WuKong, Results: {'auc': 50.0,

In [65]:
dataset = "m4a"
tasks = ["token-t-cluster4", "token-t-cluster8", "token-t-cluster16", "token-t-cluster", "token-t-cluster64"]
models = ["LR", "FM", "FFM", "AFM", "FiGNN", "WideDeep", "NFM", "DeepFM", "xDeepFM", "AutoInt", "DCN", "DCNV2", "MaskNet","FinalMLP", "EulerNet", "WuKong"]

# 收集所有结果
all_results = {}
for model in models:
    all_results[model] = {}
    for task in tasks:
        results = extract_test_results(dataset, model, task, log_dir)
        if results:
            all_results[model][task] = {
                'auc': results.get('auc', None),
                'logloss': results.get('logloss', None)
            }
        else:
            all_results[model][task] = {'auc': None, 'logloss': None}

# 按照要求的格式输出
for model in models:
    values = []
    for task in tasks:
        auc = all_results[model][task]['auc']
        if auc is not None:
            values.append(f"{auc:.2f}")
        else:
            values.append("N/A")
    
    output = f"{model} & " + ", ".join(values)
    print(output)

LR & N/A, N/A, 78.26, N/A, N/A
FM & 80.49, 80.26, 80.23, 80.14, 79.78
FFM & N/A, N/A, 80.49, N/A, N/A
AFM & 82.65, 82.98, 82.86, 82.90, 82.99
FiGNN & N/A, N/A, 81.99, N/A, N/A
WideDeep & 81.79, 81.91, 81.86, 81.75, 81.88
NFM & 82.75, 82.77, 82.79, 82.80, 82.62
DeepFM & 81.48, 81.43, 81.46, 81.11, 80.57
xDeepFM & 82.76, 82.30, 82.83, 82.76, 82.58
AutoInt & N/A, 77.91, 81.87, 80.31, 80.55
DCN & 82.52, 82.66, 82.77, 82.43, 82.35
DCNV2 & 82.86, 83.14, 83.10, 83.35, 83.23
MaskNet & 83.27, 83.57, 83.66, 83.84, 83.45
FinalMLP & N/A, 81.78, 81.83, 81.58, 81.60
EulerNet & N/A, 82.55, 82.61, 82.75, 82.64
WuKong & N/A, 50.00, N/A, N/A, 53.68


In [47]:
dataset = "m4a"
tasks = ["all-cold", "all-t-cluster16-cold"]
models = ["LR", "FM", "FFM", "AFM", "FiGNN", "WideDeep", "NFM", "DeepFM", "xDeepFM", "AutoInt", "DCN", "DCNV2", "MaskNet","FinalMLP", "EulerNet", "WuKong"]

# 收集所有结果
all_results = {}
for model in models:
    all_results[model] = {}
    for task in tasks:
        results = extract_test_results(dataset, model, task, log_dir)
        if results:
            all_results[model][task] = {
                'auc': results.get('auc', None),
                'logloss': results.get('logloss', None)
            }
        else:
            all_results[model][task] = {'auc': None, 'logloss': None}

# 按照要求的格式输出
for model in models:
    values = []
    for task in tasks:
        auc = all_results[model][task]['auc']
        logloss = all_results[model][task]['logloss']
        if auc is not None:
            values.append(f"{auc:.2f}")
        else:
            values.append("N/A")
        if logloss is not None:
            values.append(f"{logloss:.4f}")
        else:
            values.append("N/A")
    
    output = f"{model} & " + " & ".join(values)
    print(output)

LR & 69.15 & 0.4157 & 70.17 & 0.4148
FM & 58.21 & 0.4929 & 59.43 & 0.4907
FFM & N/A & N/A & N/A & N/A
AFM & 63.84 & 0.4687 & 64.92 & 0.4548
FiGNN & N/A & N/A & N/A & N/A
WideDeep & N/A & N/A & N/A & N/A
NFM & N/A & N/A & N/A & N/A
DeepFM & 60.60 & 0.5061 & 64.32 & 0.4893
xDeepFM & N/A & N/A & N/A & N/A
AutoInt & 61.39 & 0.5042 & 63.17 & 0.4774
DCN & 61.04 & 0.5150 & 64.56 & 0.5047
DCNV2 & 62.89 & 0.5235 & 65.11 & 0.4840
MaskNet & 59.08 & 0.5154 & 64.54 & 0.4767
FinalMLP & 59.29 & 0.4965 & 62.09 & 0.4690
EulerNet & 60.67 & 0.4931 & 61.27 & 0.4824
WuKong & N/A & N/A & N/A & N/A


In [57]:
dataset = "lfm2b-fil"
tasks = ["all-cold", "all-t-cluster16-cold"]
models = ["LR", "FM", "FFM", "AFM", "FiGNN", "WideDeep", "NFM", "DeepFM", "xDeepFM", "AutoInt", "DCN", "DCNV2", "MaskNet","FinalMLP", "EulerNet", "WuKong"]

# 收集所有结果
all_results = {}
for model in models:
    all_results[model] = {}
    for task in tasks:
        results = extract_test_results(dataset, model, task, log_dir)
        if results:
            all_results[model][task] = {
                'auc': results.get('auc', None),
                'logloss': results.get('logloss', None)
            }
        else:
            all_results[model][task] = {'auc': None, 'logloss': None}

# 按照要求的格式输出
for model in models:
    values = []
    for task in tasks:
        auc = all_results[model][task]['auc']
        logloss = all_results[model][task]['logloss']
        if auc is not None:
            values.append(f"{auc:.2f}")
        else:
            values.append("N/A")
        if logloss is not None:
            values.append(f"{logloss:.4f}")
        else:
            values.append("N/A")
    
    output = f"{model} & " + " & ".join(values)
    print(output)

LR & 54.05 & 0.7505 & 56.50 & 0.7286
FM & N/A & N/A & N/A & N/A
FFM & N/A & N/A & N/A & N/A
AFM & 52.04 & 0.9047 & 56.80 & 0.8233
FiGNN & N/A & N/A & N/A & N/A
WideDeep & N/A & N/A & N/A & N/A
NFM & N/A & N/A & N/A & N/A
DeepFM & N/A & N/A & N/A & N/A
xDeepFM & N/A & N/A & N/A & N/A
AutoInt & 55.29 & 9.3026 & 54.18 & 10.7074
DCN & 60.44 & 1.0249 & 58.94 & 1.2274
DCNV2 & 56.00 & 1.4961 & 59.99 & 1.3548
MaskNet & 51.11 & 0.9479 & 59.17 & 0.8813
FinalMLP & 50.41 & 0.9099 & 56.38 & 0.8140
EulerNet & 53.82 & 1.0539 & 60.60 & 0.9780
WuKong & N/A & N/A & N/A & N/A


In [56]:
dataset = "lfm1b-fil"
tasks = ["all", "all-a", "all-cluster16"]
models = ["LR", "FM", "FFM", "AFM", "FiGNN", "WideDeep", "NFM", "DeepFM", "xDeepFM", "AutoInt", "DCN", "DCNV2", "MaskNet","FinalMLP", "EulerNet", "WuKong"]

# 收集所有结果
all_results = {}
for model in models:
    all_results[model] = {}
    for task in tasks:
        results = extract_test_results(dataset, model, task, log_dir)
        if results and 'auc' in results:
            all_results[model][task] = results['auc']
        else:
            all_results[model][task] = None

# 按照要求的格式输出
for model in models:
    auc_values = []
    for task in tasks:
        auc = all_results[model][task]
        if auc is not None:
            auc_values.append(f"{auc:.2f}")
        else:
            auc_values.append("N/A")
    
    output = f"{model} & " + " & ".join(auc_values)
    print(output)

LR & 80.70 & 80.70 & 80.66
FM & 84.34 & 84.29 & 84.11
FFM & 85.53 & 85.48 & N/A
AFM & 86.47 & 86.45 & 86.75
FiGNN & 85.49 & 85.72 & 85.91
WideDeep & 85.58 & 85.62 & N/A
NFM & 85.78 & 86.00 & 86.50
DeepFM & 83.72 & 83.78 & 83.40
xDeepFM & 85.70 & 85.87 & 86.29
AutoInt & 82.14 & 84.96 & 83.86
DCN & 86.75 & 86.76 & N/A
DCNV2 & 86.81 & 86.81 & N/A
MaskNet & 86.98 & 87.00 & 87.19
FinalMLP & 86.40 & 86.36 & N/A
EulerNet & 86.83 & 86.88 & N/A
WuKong & 87.33 & 50.00 & N/A


In [ ]:
dataset = "lfm2b-fil"
task_name = "id-t-cluster16"
models = ["LR", "FM", "FFM", "AFM", "FiGNN", "WideDeep", "NFM", "DeepFM", "xDeepFM", "AutoInt", "DCN", "DCNV2", "MaskNet","FinalMLP", "EulerNet", "WuKong"]
for model in models:
    results = extract_test_results(dataset, model, task_name, log_dir)
    if results:
        print(f"Model: {model}, Results: {results}")
    else:
        print(f"Model: {model}, No results found.")

Model: LR, Results: {'auc': 81.21, 'logloss': 0.4395, 'ndcg': 0.6469}
Model: FM, Results: {'auc': 84.82, 'logloss': 0.4433, 'ndcg': 0.5431}
未在任何匹配的log文件中找到测试结果
Model: FFM, No results found.
Model: AFM, Results: {'auc': 85.83, 'logloss': 0.3997, 'ndcg': 1.0}
Model: FiGNN, Results: {'auc': 85.74, 'logloss': 0.3964, 'ndcg': 1.0}
Model: WideDeep, Results: {'auc': 86.0, 'logloss': 0.3953, 'ndcg': 0.8365}
Model: NFM, Results: {'auc': 86.02, 'logloss': 0.3927, 'ndcg': 1.0}
Model: DeepFM, Results: {'auc': 83.86, 'logloss': 0.4564, 'ndcg': 0.8611}
Model: xDeepFM, Results: {'auc': 85.84, 'logloss': 0.3893, 'ndcg': 1.0}
未在任何匹配的log文件中找到测试结果
Model: AutoInt, No results found.
Model: DCN, Results: {'auc': 86.62, 'logloss': 0.3829, 'ndcg': 1.0}
Model: DCNV2, Results: {'auc': 86.89, 'logloss': 0.3795, 'ndcg': 1.0}
Model: MaskNet, Results: {'auc': 86.68, 'logloss': 0.3802, 'ndcg': 1.0}
Model: FinalMLP, Results: {'auc': 86.53, 'logloss': 0.3874, 'ndcg': 1.0}
未在任何匹配的log文件中找到测试结果
Model: EulerNet, No result

In [51]:
dataset = "lfm2b-fil"
tasks = ["idonly", "id-a-t", "id-t-cluster16"]
models = ["LR", "FM", "FFM", "AFM", "FiGNN", "WideDeep", "NFM", "DeepFM", "xDeepFM", "AutoInt", "DCN", "DCNV2", "MaskNet","FinalMLP", "EulerNet", "WuKong"]

# 收集所有结果
all_results = {}
for model in models:
    all_results[model] = {}
    for task in tasks:
        results = extract_test_results(dataset, model, task, log_dir)
        if results and 'auc' in results:
            all_results[model][task] = results['auc']
        else:
            all_results[model][task] = None

# 按照要求的格式输出
for model in models:
    auc_values = []
    for task in tasks:
        auc = all_results[model][task]
        if auc is not None:
            auc_values.append(f"{auc:.2f}")
        else:
            auc_values.append("N/A")
    
    output = f"{model} & " + " & ".join(auc_values)
    print(output)

LR & 81.27 & N/A & 81.21
FM & 85.17 & 85.25 & 84.82
FFM & 84.06 & N/A & 83.97
AFM & 85.12 & 85.47 & 85.83
FiGNN & 84.31 & N/A & 85.74
WideDeep & 84.02 & 84.84 & 86.00
NFM & 83.32 & 85.85 & 86.02
DeepFM & 83.81 & 84.06 & 83.86
xDeepFM & 82.03 & 85.52 & 85.84
AutoInt & 84.63 & N/A & N/A
DCN & 85.22 & 85.90 & 86.62
DCNV2 & 85.60 & 86.58 & 86.89
MaskNet & 84.78 & 86.57 & 86.68
FinalMLP & 85.79 & 86.29 & 86.53
EulerNet & 85.01 & 85.64 & 86.53
WuKong & 83.64 & 50.00 & 50.00


In [52]:
dataset = "lfm2b-fil"
tasks = ["token", "token-a-t", "token-t-cluster16"]
models = ["LR", "FM", "FFM", "AFM", "FiGNN", "WideDeep", "NFM", "DeepFM", "xDeepFM", "AutoInt", "DCN", "DCNV2", "MaskNet","FinalMLP", "EulerNet", "WuKong"]

# 收集所有结果
all_results = {}
for model in models:
    all_results[model] = {}
    for task in tasks:
        results = extract_test_results(dataset, model, task, log_dir)
        if results and 'auc' in results:
            all_results[model][task] = results['auc']
        else:
            all_results[model][task] = None

# 按照要求的格式输出
for model in models:
    auc_values = []
    for task in tasks:
        auc = all_results[model][task]
        if auc is not None:
            auc_values.append(f"{auc:.2f}")
        else:
            auc_values.append("N/A")
    
    output = f"{model} & " + " & ".join(auc_values)
    print(output)

LR & N/A & 81.24 & 81.20
FM & 85.26 & 85.33 & 84.82
FFM & N/A & 84.22 & 84.28
AFM & 85.41 & 85.89 & 86.26
FiGNN & N/A & 85.94 & 85.91
WideDeep & 84.50 & 85.87 & 86.17
NFM & 85.08 & 86.19 & 86.38
DeepFM & 84.11 & 84.29 & 83.85
xDeepFM & 85.21 & 85.60 & 86.12
AutoInt & 83.26 & 80.19 & N/A
DCN & 85.87 & 86.64 & 86.87
DCNV2 & 86.54 & 86.91 & 87.10
MaskNet & 85.86 & 87.00 & 87.07
FinalMLP & 85.55 & 86.53 & 86.82
EulerNet & 85.64 & 86.55 & 86.81
WuKong & 84.66 & 50.00 & 50.00


In [58]:
dataset = "lfm2b-fil"
tasks = ["all", "all-a-t", "all-t-cluster16"]
models = ["LR", "FM", "FFM", "AFM", "FiGNN", "WideDeep", "NFM", "DeepFM", "xDeepFM", "AutoInt", "DCN", "DCNV2", "MaskNet","FinalMLP", "EulerNet", "WuKong"]

# 收集所有结果
all_results = {}
for model in models:
    all_results[model] = {}
    for task in tasks:
        results = extract_test_results(dataset, model, task, log_dir)
        if results and 'auc' in results:
            all_results[model][task] = results['auc']
        else:
            all_results[model][task] = None

# 按照要求的格式输出
for model in models:
    auc_values = []
    for task in tasks:
        auc = all_results[model][task]
        if auc is not None:
            auc_values.append(f"{auc:.2f}")
        else:
            auc_values.append("N/A")
    
    output = f"{model} & " + " & ".join(auc_values)
    print(output)

LR & 81.27 & 81.23 & N/A
FM & 85.32 & 85.28 & 84.79
FFM & 84.22 & 84.25 & N/A
AFM & 85.44 & 85.91 & 86.28
FiGNN & 84.96 & 85.86 & 85.92
WideDeep & 85.38 & 85.63 & N/A
NFM & 85.10 & 86.19 & 86.42
DeepFM & 84.11 & 84.24 & 83.93
xDeepFM & 85.21 & 86.07 & 86.13
AutoInt & 76.54 & 68.59 & 70.88
DCN & 86.43 & 86.73 & N/A
DCNV2 & 86.47 & 87.01 & N/A
MaskNet & 85.94 & 87.01 & 87.11
FinalMLP & 86.25 & 86.55 & N/A
EulerNet & 86.44 & 86.56 & N/A
WuKong & 85.10 & 50.00 & N/A


In [59]:
dataset = "lfm1b-fil"
tasks = ["idonly", "id-a", "id-cluster16"]
models = ["LR", "FM", "FFM", "AFM", "FiGNN", "WideDeep", "NFM", "DeepFM", "xDeepFM", "AutoInt", "DCN", "DCNV2", "MaskNet","FinalMLP", "EulerNet", "WuKong"]

# 收集所有结果
all_results = {}
for model in models:
    all_results[model] = {}
    for task in tasks:
        results = extract_test_results(dataset, model, task, log_dir)
        if results and 'auc' in results:
            all_results[model][task] = results['auc']
        else:
            all_results[model][task] = None

# 按照要求的格式输出
for model in models:
    auc_values = []
    for task in tasks:
        auc = all_results[model][task]
        if auc is not None:
            auc_values.append(f"{auc:.2f}")
        else:
            auc_values.append("N/A")
    
    output = f"{model} & " + " & ".join(auc_values)
    print(output)

LR & N/A & N/A & 80.74
FM & 84.48 & 84.67 & 84.17
FFM & 83.34 & N/A & 83.83
AFM & 84.90 & 84.78 & 85.17
FiGNN & N/A & N/A & 84.84
WideDeep & 84.58 & 84.78 & 85.45
NFM & 82.24 & 83.87 & 85.23
DeepFM & 83.43 & 83.16 & 83.37
xDeepFM & 83.43 & 83.79 & 85.18
AutoInt & 84.80 & N/A & 84.49
DCN & 85.37 & 85.23 & 86.18
DCNV2 & 85.36 & 85.55 & 86.21
MaskNet & 83.85 & 85.23 & 85.93
FinalMLP & 85.05 & 85.05 & 85.95
EulerNet & 85.40 & 85.57 & 85.96
WuKong & 82.72 & N/A & 50.00


In [55]:
dataset = "lfm1b-fil"
tasks = ["token", "token-a", "token-cluster16"]
models = ["LR", "FM", "FFM", "AFM", "FiGNN", "WideDeep", "NFM", "DeepFM", "xDeepFM", "AutoInt", "DCN", "DCNV2", "MaskNet","FinalMLP", "EulerNet", "WuKong"]

# 收集所有结果
all_results = {}
for model in models:
    all_results[model] = {}
    for task in tasks:
        results = extract_test_results(dataset, model, task, log_dir)
        if results and 'auc' in results:
            all_results[model][task] = results['auc']
        else:
            all_results[model][task] = None

# 按照要求的格式输出
for model in models:
    auc_values = []
    for task in tasks:
        auc = all_results[model][task]
        if auc is not None:
            auc_values.append(f"{auc:.2f}")
        else:
            auc_values.append("N/A")
    
    output = f"{model} & " + " & ".join(auc_values)
    print(output)

LR & 80.78 & 80.73 & 80.74
FM & 84.61 & 84.55 & 84.22
FFM & 83.98 & 83.95 & 83.98
AFM & 84.62 & 85.23 & 85.49
FiGNN & 84.41 & 84.89 & 85.16
WideDeep & 84.88 & 84.99 & 85.57
NFM & 83.82 & 85.48 & 85.55
DeepFM & 83.37 & 83.68 & 83.56
xDeepFM & 84.56 & 85.25 & 85.24
AutoInt & 82.51 & 81.46 & 83.18
DCN & 85.87 & 85.93 & 86.41
DCNV2 & 85.81 & 86.00 & 86.47
MaskNet & 85.05 & 86.07 & 86.29
FinalMLP & 85.58 & 85.67 & 86.15
EulerNet & 85.75 & 85.73 & 86.25
WuKong & 82.97 & N/A & 50.00


In [54]:
dataset = "lfm1b-fil"
tasks = ["all", "all-a", "all-cluster16"]
models = ["LR", "FM", "FFM", "AFM", "FiGNN", "WideDeep", "NFM", "DeepFM", "xDeepFM", "AutoInt", "DCN", "DCNV2", "MaskNet","FinalMLP", "EulerNet", "WuKong"]

# 收集所有结果
all_results = {}
for model in models:
    all_results[model] = {}
    for task in tasks:
        results = extract_test_results(dataset, model, task, log_dir)
        if results and 'auc' in results:
            all_results[model][task] = results['auc']
        else:
            all_results[model][task] = None

# 按照要求的格式输出
for model in models:
    auc_values = []
    for task in tasks:
        auc = all_results[model][task]
        if auc is not None:
            auc_values.append(f"{auc:.2f}")
        else:
            auc_values.append("N/A")
    
    output = f"{model} & " + " & ".join(auc_values)
    print(output)

LR & 80.70 & 80.70 & 80.66
FM & 84.34 & 84.29 & 84.11
FFM & 85.53 & 85.48 & N/A
AFM & 86.47 & 86.45 & 86.75
FiGNN & 85.49 & 85.72 & 85.91
WideDeep & 85.58 & 85.62 & N/A
NFM & 85.78 & 86.00 & 86.50
DeepFM & 83.72 & 83.78 & 83.40
xDeepFM & 85.70 & 85.87 & 86.29
AutoInt & 82.14 & 84.96 & 83.86
DCN & 86.75 & 86.76 & N/A
DCNV2 & 86.81 & 86.81 & N/A
MaskNet & 86.98 & 87.00 & 87.19
FinalMLP & 86.40 & 86.36 & N/A
EulerNet & 86.83 & 86.88 & N/A
WuKong & 87.33 & 50.00 & N/A


In [45]:
dataset = "m4a"
tasks = ["token-mfcc-t", "token-msclap-t", "token-mulan-t", "token-t-mulan16", "token-t-cluster16"]
models = ["LR", "FM", "FFM", "AFM", "FiGNN", "WideDeep", "NFM", "DeepFM", "xDeepFM", "AutoInt", "DCN", "DCNV2", "MaskNet","FinalMLP", "EulerNet", "WuKong"]

# 收集所有结果
all_results = {}
for model in models:
    all_results[model] = {}
    for task in tasks:
        results = extract_test_results(dataset, model, task, log_dir)
        if results and 'auc' in results:
            all_results[model][task] = results['auc']
        else:
            all_results[model][task] = None

# 按照要求的格式输出
for model in models:
    auc_values = []
    for task in tasks:
        auc = all_results[model][task]
        if auc is not None:
            auc_values.append(f"{auc:.2f}")
        else:
            auc_values.append("N/A")
    
    output = f"{model} & " + " & ".join(auc_values)
    print(output)

LR & N/A & N/A & N/A & 78.30 & 78.26
FM & 80.27 & 80.34 & 80.81 & 80.60 & 80.23
FFM & N/A & N/A & N/A & 81.12 & 80.49
AFM & 81.52 & 81.57 & 82.65 & 82.69 & 82.86
FiGNN & N/A & N/A & N/A & 82.00 & 81.99
WideDeep & N/A & 81.49 & 81.78 & N/A & 81.86
NFM & 81.73 & 82.37 & 82.66 & 82.63 & 82.79
DeepFM & 81.36 & 81.59 & 81.86 & N/A & 81.46
xDeepFM & N/A & 82.00 & 82.65 & N/A & 82.83
AutoInt & N/A & 81.38 & 81.38 & N/A & 81.87
DCN & N/A & 81.58 & 82.67 & N/A & 82.77
DCNV2 & 81.89 & 82.51 & 82.92 & 82.78 & 83.10
MaskNet & 82.54 & 83.05 & 83.45 & 83.30 & 83.66
FinalMLP & N/A & 80.33 & 80.33 & N/A & 81.83
EulerNet & N/A & 82.13 & 82.80 & N/A & 82.61
WuKong & N/A & N/A & 83.18 & N/A & N/A
